# 🤖 Cortex Analytics Orchestrator - Interactive Demo

## Natural Language Analytics Interface

Ask questions in plain English and watch the multi-agent system orchestrate the workflow!

---

## 📦 Setup - Run Once

In [ ]:
# Install required packages (run once)
# !pip install snowflake-connector-python snowflake-snowpark-python plotly ipywidgets pandas

In [1]:
import snowflake.connector
from snowflake.snowpark import Session
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from datetime import datetime
import json

print("✅ All packages loaded successfully!")

✅ All packages loaded successfully!


## 🔐 Snowflake Connection

In [2]:
# Configure your Snowflake connection
SNOWFLAKE_CONFIG = {
    'account':'wkswaox-lt08934', 
    'user':'Lima717',
    'password':'Easy2snowflake!',
    'role':'ACCOUNTADMIN',
    'warehouse':'COMPUTE_WH',
    'database':'CAMPAIGN_ANALYTICS',
    'schema':'GENERATED_DATA'
}

# Create Snowflake session
session = Session.builder.configs(SNOWFLAKE_CONFIG).create()
print(f"✅ Connected to Snowflake: {SNOWFLAKE_CONFIG['account']}")
print(f"📊 Using database: {SNOWFLAKE_CONFIG['database']}.{SNOWFLAKE_CONFIG['schema']}")

✅ Connected to Snowflake: wkswaox-lt08934
📊 Using database: CAMPAIGN_ANALYTICS.GENERATED_DATA


## 🤖 Load Multi-Agent System

This cell loads your CortexAnalyst, VisualizationAgent, and MultiAgentExecutor

In [23]:
import re

class CortexAnalyst:
    """Cortex Analyst simulator using CORTEX.COMPLETE"""
    
    def __init__(self, session, semantic_model_stage):
        self.session = session
        self.semantic_model = self._load_semantic_model(semantic_model_stage)
        
        # IMPROVED: Use strategic context instead of first 2500 chars
        self.schema_context = self._build_context()
        
        print(f"✅ Loaded semantic model ({len(self.semantic_model)} chars)")
        print(f"✅ Built context ({len(self.schema_context)} chars)")
        
    def _load_semantic_model(self, stage_path):
        """Load semantic model from Snowflake stage"""
        # Create file format
        self.session.sql("""
            CREATE FILE FORMAT IF NOT EXISTS yaml_format
            TYPE = 'CSV'
            FIELD_DELIMITER = NONE
            RECORD_DELIMITER = NONE
        """).collect()
        
        # Read file
        result = self.session.sql(f"""
            SELECT t.$1 AS content
            FROM {stage_path} (FILE_FORMAT => yaml_format) t
        """).collect()
        
        return "\n".join([row['CONTENT'] for row in result])
    
    def _build_context(self):
        """Build strategic context for better generalization"""
        
        # Get critical sections
        header = self.semantic_model[:800]  # Name, physical columns
        
        # Extract verified queries
        verified_start = self.semantic_model.find('verified_queries:')
        verified_end = self.semantic_model.find('tables:', verified_start)
        verified = self.semantic_model[verified_start:verified_end] if verified_start != -1 else ""
        
        # Get first table definition
        tables_start = self.semantic_model.find('tables:')
        first_table = self.semantic_model[tables_start:tables_start+2000] if tables_start != -1 else ""
        
        return f"{header}\n\n{verified}\n\n{first_table}"
    
    def _clean_sql(self, text):
        """Remove markdown and formatting from SQL"""
        text = re.sub(r'```sql\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(r'```\s*', '', text)
        text = text.strip()
        
        # Check for explanations
        explanation_keywords = ['to answer', 'this query', 'here is', 'we need', 'first']
        if any(text.lower().startswith(keyword) for keyword in explanation_keywords):
            # Try to extract SQL
            match = re.search(r'((?:WITH|SELECT).*?)(?:;|\Z)', text, re.IGNORECASE | re.DOTALL)
            if match:
                text = match.group(1)
            else:
                raise ValueError("LLM returned explanation without SQL")
        
        return text.rstrip(';').strip()
    
    def ask(self, question):
        # Simplified prompt
        prompt = f"""Generate SQL query for Snowflake.

    Schema info:
    {self.schema_context[:1000]}

    Rules:
    - Return ONLY the SQL query
    - No explanations
    - No markdown
    - No extra text

    Question: {question}

    SQL:"""
        
        result = self.session.sql(f"""
            SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large', '{prompt.replace("'", "''")}') AS sql_text
        """).collect()
        
        raw_sql = result[0]['SQL_TEXT']
        
        # Debug: print what we got
        print(f"   [DEBUG] Raw LLM output: {raw_sql[:200]}...")
        
        clean_sql = self._clean_sql(raw_sql)
        
        print(f"   [DEBUG] Clean SQL: {clean_sql[:200]}...")
        
        try:
            data = self.session.sql(clean_sql).collect()
            results = []
            for row in data:
                row_dict = row.asDict()
                converted_row = {}
                for key, value in row_dict.items():
                    if isinstance(value, str):
                        try:
                            if '.' in value:
                                converted_row[key] = float(value)
                            else:
                                converted_row[key] = int(value)
                        except (ValueError, AttributeError):
                            converted_row[key] = value
                    else:
                        converted_row[key] = value
                results.append(converted_row)
            return {'success': True, 'sql': clean_sql, 'results': results, 'row_count': len(results)}
        except Exception as e:
            return {'success': False, 'sql': clean_sql, 'results': [], 'error': str(e)}

print("✅ CortexAnalyst class defined")

✅ CortexAnalyst class defined


In [18]:
class CortexAnalyst:
    def __init__(self, session, semantic_model_stage_path: str):
        self.session = session
        self.semantic_model_path = semantic_model_stage_path
        self.semantic_model = self._load_semantic_model()
        self.context = self._build_context()
        
    def _load_semantic_model(self):
        """Load semantic model from Snowflake stage"""
        try:
            result = self.session.sql(f"SELECT $1 FROM {self.semantic_model_path}").collect()
            if result and len(result) > 0:
                model_text = result[0][0]
                print(f"✅ Loaded semantic model ({len(model_text)} characters)")
                return model_text
            else:
                print("⚠️ Warning: Empty semantic model")
                return ""
        except Exception as e:
            print(f"⚠️ Warning: Could not load semantic model: {e}")
            return ""
    
    def _build_context(self):
        """Build context for LLM from semantic model"""
        if not self.semantic_model:
            return ""
        
        # Extract table information from semantic model
        context_parts = []
        
        # Add instruction
        context_parts.append("Generate SQL queries using ONLY the tables and columns defined below.")
        context_parts.append("\nAvailable tables and columns:\n")
        
        # Try to extract table info from semantic model
        if "V_FACT_SFMC_SEND_PERFORMANCE_TRACKING" in self.semantic_model:
            context_parts.append("""
Table: V_FACT_SFMC_SEND_PERFORMANCE_TRACKING
Dimensions: BUSINESSUNIT (market), SENDID, SENDID_COUNTRY_SK
Time: SENDDATE
Facts: SENDS, BOUNCES, OPENS, UNIQUEOPENS, CLICKS, UNIQUECLICKS
Calculations:
  - OPEN_RATE = (UNIQUEOPENS / NULLIF(SENDS - BOUNCES, 0)) * 100
  - CLICK_RATE = (UNIQUECLICKS / NULLIF(UNIQUEOPENS, 0)) * 100
""")
        
        return "\n".join(context_parts)
    
    def ask(self, question: str) -> dict:
        """Generate SQL from natural language question using Cortex"""
        try:
            # Build the prompt with context
            prompt = f"""You are a SQL expert for Snowflake. Generate a SQL query based on this question.

{self.context}

IMPORTANT RULES:
1. Use ONLY the tables and columns listed above
2. NEVER make up table names
3. For open rates, calculate as: (UNIQUEOPENS / NULLIF(SENDS - BOUNCES, 0)) * 100
4. For click rates, calculate as: (UNIQUECLICKS / NULLIF(UNIQUEOPENS, 0)) * 100
5. Use BUSINESSUNIT column for market/region analysis
6. Return ONLY the SQL query, no explanations

Question: {question}

SQL Query:"""
            
            # Call Cortex Complete
            llm_result = self.session.sql(f"""
                SELECT SNOWFLAKE.CORTEX.COMPLETE(
                    'mistral-large',
                    $${prompt}$$
                )
            """).collect()
            
            if not llm_result:
                return {
                    'success': False,
                    'error': 'No response from LLM'
                }
            
            raw_sql = llm_result[0][0]
            
            # Clean the SQL
            sql = self._clean_sql(raw_sql)
            
            if not sql:
                return {
                    'success': False,
                    'error': 'Could not extract valid SQL from LLM response'
                }
            
            print(f"📝 Generated SQL: {sql[:100]}...")
            
            # Execute the SQL
            results = self.session.sql(sql).collect()
            
            return {
                'success': True,
                'sql': sql,
                'results': results,
                'row_count': len(results)
            }
            
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Error: {error_msg}")
            return {
                'success': False,
                'error': error_msg,
                'sql': sql if 'sql' in locals() else None
            }
    
    def _clean_sql(self, raw_sql: str) -> str:
        """Clean and extract SQL from LLM response"""
        sql = raw_sql.strip()
        
        # Remove markdown code blocks
        if '```sql' in sql:
            sql = sql.split('```sql')[1].split('```')[0].strip()
        elif '```' in sql:
            sql = sql.split('```')[1].split('```')[0].strip()
        
        # Remove any "Reference:" lines or explanatory text
        lines = []
        for line in sql.split('\n'):
            line_stripped = line.strip()
            if line_stripped and not any(skip in line_stripped.lower() for skip in 
                ['reference:', 'note:', 'explanation:', 'this query', 'the query']):
                lines.append(line)
        
        sql = '\n'.join(lines).strip()
        
        # Basic validation
        if not sql.upper().startswith('SELECT'):
            # Try to find SELECT statement
            for line in sql.split('\n'):
                if line.strip().upper().startswith('SELECT'):
                    sql = '\n'.join(sql.split('\n')[sql.split('\n').index(line):])
                    break
        
        return sql

print("✅ CortexAnalyst class loaded")

✅ CortexAnalyst class loaded


In [6]:
# Paste your VisualizationAgent class here
class VisualizationAgent:
    def create_chart(self, df: pd.DataFrame, query_type: str = 'GENERAL'):
        """Create appropriate visualization based on data and query type"""
        try:
            if df.empty:
                return None
            
            # Determine chart type based on query type and data
            if query_type in ['COMPARISON', 'GENERAL'] and len(df.columns) == 2:
                # Bar chart for comparisons
                fig = px.bar(
                    df,
                    x=df.columns[0],
                    y=df.columns[1],
                    title=f"{df.columns[1]} by {df.columns[0]}",
                    color=df.columns[1],
                    color_continuous_scale="Blues"
                )
                fig.update_layout(showlegend=False, height=500)
                
            elif query_type == 'RANKING':
                # Horizontal bar chart for rankings
                fig = px.bar(
                    df,
                    x=df.columns[1],
                    y=df.columns[0],
                    orientation='h',
                    title=f"Top {len(df)} by {df.columns[1]}",
                    color=df.columns[1],
                    color_continuous_scale="Viridis"
                )
                fig.update_layout(showlegend=False, height=400)
                
            elif query_type == 'TREND' and len(df.columns) >= 2:
                # Line chart for trends
                fig = px.line(
                    df,
                    x=df.columns[0],
                    y=df.columns[1],
                    title=f"{df.columns[1]} Trend",
                    markers=True
                )
                fig.update_traces(line_color='#1F4788', line_width=2)
                fig.update_layout(height=500)
                
            else:
                # Default: simple bar chart
                fig = px.bar(
                    df,
                    x=df.columns[0],
                    y=df.columns[1] if len(df.columns) > 1 else df.columns[0],
                    title="Query Results"
                )
                fig.update_layout(height=500)
            
            return fig
            
        except Exception as e:
            print(f"⚠️ Could not create visualization: {e}")
            return None

print("✅ VisualizationAgent class loaded")

✅ VisualizationAgent class loaded


In [8]:
# Paste your MultiAgentExecutor class here
class MultiAgentExecutor:
    def __init__(self, session, semantic_model_path: str):
        self.session = session
        self.analyst = CortexAnalyst(session, semantic_model_path)
        self.visualizer = VisualizationAgent()
        
    def _classify_query(self, question: str) -> str:
        """Classify the type of query"""
        question_lower = question.lower()
        
        if any(word in question_lower for word in ['top', 'bottom', 'highest', 'lowest', 'best', 'worst']):
            return 'RANKING'
        elif any(word in question_lower for word in ['trend', 'over time', 'time series', 'history']):
            return 'TREND'
        elif any(word in question_lower for word in ['compare', 'by', 'each', 'per']):
            return 'COMPARISON'
        elif any(word in question_lower for word in ['total', 'sum', 'count', 'average', 'avg']):
            return 'AGGREGATION'
        else:
            return 'GENERAL'
    
    def execute(self, question: str, show_viz: bool = True) -> dict:
        """Execute the full multi-agent workflow"""
        # Step 1: Classify query
        query_type = self._classify_query(question)
        
        # Step 2: Get SQL and data from analyst
        data_result = self.analyst.ask(question)
        
        if not data_result['success']:
            return {
                'success': False,
                'query_type': query_type,
                'error': data_result['error']
            }
        
        # Step 3: Convert to DataFrame
        df = pd.DataFrame(data_result['results'])
        
        # Step 4: Create visualization
        visualization = None
        if show_viz and not df.empty:
            visualization = self.visualizer.create_chart(df, query_type)
        
        # Step 5: Generate insights
        insights = self._generate_insights(df, query_type)
        
        return {
            'success': True,
            'query_type': query_type,
            'sql': data_result['sql'],
            'data': data_result['results'],
            'dataframe': df,
            'row_count': len(df),
            'visualization': visualization,
            'insights': insights
        }
    
    def _generate_insights(self, df: pd.DataFrame, query_type: str) -> str:
        """Generate simple insights from the data"""
        if df.empty:
            return "No data returned"
        
        insights = []
        
        if query_type == 'RANKING' and len(df) > 0:
            top_item = df.iloc[0, 0]
            insights.append(f"Top performer: {top_item}")
            
        if len(df.columns) >= 2 and df.iloc[:, 1].dtype in ['int64', 'float64']:
            total = df.iloc[:, 1].sum()
            avg = df.iloc[:, 1].mean()
            insights.append(f"Total: {total:,.0f}")
            insights.append(f"Average: {avg:,.2f}")
        
        return " | ".join(insights) if insights else f"Retrieved {len(df)} rows"

print("✅ MultiAgentExecutor class loaded")

✅ MultiAgentExecutor class loaded


## 🎯 Initialize the System

In [24]:
# Initialize the executor with your semantic model path
SEMANTIC_MODEL_PATH = '@SEMANTIC_MODELS/marketing_semantic_model.yaml'

executor = MultiAgentExecutor(session, SEMANTIC_MODEL_PATH)

print("🎉 System initialized and ready!")
print("\n📋 Sample questions you can ask:")
print("   • Show me email open rates by market for the past month")
print("   • What are the top 5 markets by sends?")
print("   • Total sends this month")
print("   • Compare open rates across all markets")

✅ Loaded semantic model (10484 chars)
✅ Built context (5535 chars)
🎉 System initialized and ready!

📋 Sample questions you can ask:
   • Show me email open rates by market for the past month
   • What are the top 5 markets by sends?
   • Total sends this month
   • Compare open rates across all markets


---

# 🎮 Interactive Query Interface

## Ask Your Question Below! 👇

In [ ]:
# ============================================================
# STEP 1: Create all widgets first
# ============================================================

question_input = widgets.Textarea(
    value='Show me email open rates by market for the past month',
    placeholder='Type your question here...',
    description='Question:',
    layout=widgets.Layout(width='90%', height='80px'),
    style={'description_width': '100px'}
)

execute_button = widgets.Button(
    description='🚀 Execute Query',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output()

# ============================================================
# STEP 2: Define the execution function
# ============================================================

def execute_query_and_display(question):
    """Execute query and display all results"""
    
    # Clear previous output
    output_area.clear_output()
    
    with output_area:
        if not question.strip():
            print("❌ Please enter a question")
            return
        
        # Show processing
        display(HTML(f"""
        <div style='padding: 20px; background: #f0f8ff; border-radius: 10px; border-left: 5px solid #1F4788;'>
            <h3>🤖 Multi-Agent System Processing...</h3>
            <p style='color: #666;'>Question: <strong>{question}</strong></p>
        </div>
        """))
        
        try:
            # Execute query
            result = executor.execute(question, show_viz=True)
            
            if result['success']:
                # Query classification
                display(HTML(f"""
                <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; margin: 10px 0;'>
                    <strong>📋 Query Classification:</strong> {result['query_type']}
                </div>
                """))
                
                # Generated SQL
                display(Markdown("### 🔍 Generated SQL"))
                display(HTML(f"""
                <div style='background: #f5f5f5; padding: 15px; border-radius: 8px; overflow-x: auto;'>
                    <pre style='margin: 0;'>{result['sql']}</pre>
                </div>
                """))
                
                # Insights
                display(HTML(f"""
                <div style='padding: 15px; background: #fff3e0; border-radius: 8px; margin: 10px 0;'>
                    <strong>💡 Insights:</strong> {result['insights']}
                </div>
                """))
                
                # Results table
                display(Markdown("### 📊 Results"))
                display(HTML(f"<p><strong>Retrieved {result['row_count']} rows</strong></p>"))
                
                if result['row_count'] > 0:
                    display_df = result['dataframe'].head(10)
                    display(display_df.style.set_properties(**{
                        'background-color': '#f9f9f9',
                        'border-color': '#ddd',
                        'text-align': 'left'
                    }).set_table_styles([{
                        'selector': 'th',
                        'props': [('background-color', '#1F4788'), ('color', 'white'), 
                                 ('font-weight', 'bold'), ('text-align', 'left')]
                    }]))
                    
                    if result['row_count'] > 10:
                        display(HTML(f"<p><em>Showing first 10 of {result['row_count']} rows</em></p>"))
                
                # Visualization
                if result['visualization']:
                    display(Markdown("### 📈 Visualization"))
                    result['visualization'].show()
                
                # Success message
                display(HTML("""
                <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; margin: 20px 0;'>
                    <h3 style='color: #2e7d32; margin: 0;'>✅ Query Completed Successfully!</h3>
                </div>
                """))
                
            else:
                # Error
                display(HTML(f"""
                <div style='padding: 20px; background: #ffebee; border-radius: 10px; border-left: 5px solid #c62828;'>
                    <h3 style='color: #c62828;'>❌ Error</h3>
                    <p>{result['error']}</p>
                </div>
                """))
        
        except Exception as e:
            display(HTML(f"""
            <div style='padding: 20px; background: #ffebee; border-radius: 10px; border-left: 5px solid #c62828;'>
                <h3 style='color: #c62828;'>❌ Unexpected Error</h3>
                <p>{str(e)}</p>
            </div>
            """))

# ============================================================
# STEP 3: Connect main button
# ============================================================

def on_main_button_click(b):
    """Handler for main execute button"""
    question = question_input.value.strip()
    execute_query_and_display(question)

execute_button.on_click(on_main_button_click)

# ============================================================
# STEP 4: Display main interface
# ============================================================

display(HTML("<h2>🎮 Interactive Query Interface</h2>"))
display(HTML("<p>Ask your question below:</p>"))

display(widgets.VBox([
    question_input,
    execute_button,
    output_area
]))

# ============================================================
# STEP 5: Create test buttons
# ============================================================

display(HTML("<br><h3>🎯 Quick Test Queries</h3>"))
display(HTML("<p>Click these buttons to test common queries:</p>"))

test_queries = [
    "Show me email open rates by market for the past month",
    "What are the top 5 markets by sends?",
    "Total sends this month",
    "Compare click rates across all markets"
]

test_buttons = []

for query in test_queries:
    btn = widgets.Button(
        description=query[:40] + '...' if len(query) > 40 else query,
        layout=widgets.Layout(width='95%', height='40px'),
        button_style='info',
        tooltip=query  # Show full query on hover
    )
    
    # Create a closure to capture the query value
    def make_handler(q):
        def handler(b):
            question_input.value = q  # Update text box
            execute_query_and_display(q)  # Execute immediately
        return handler
    
    btn.on_click(make_handler(query))
    test_buttons.append(btn)

display(widgets.VBox(test_buttons))

print("✅ Interface loaded and ready!")
print("💡 Click any button above or type your own question")


✅ Interface loaded and ready!
💡 Click any button above or type your own question


---

## 🎯 Quick Test Queries

Click these buttons to test common queries:

In [31]:
# Quick test buttons
test_queries = [
    "Show me email open rates by market for the past month",
    "What are the top 5 markets by sends?",
    "Total sends this month",
    "Compare click rates across all markets"
]

def create_test_button(query):
    button = widgets.Button(
        description=query[:40] + '...' if len(query) > 40 else query,
        layout=widgets.Layout(width='100%', height='40px'),
        button_style='info'
    )
    
    def on_click(b):
        # Set the question
        question_input.value = query
        
        # Call the execution logic directly
        with output_area:
            output_area.clear_output()
            
            if not query.strip():
                print("❌ Please enter a question")
                return
            
            # Show processing animation
            display(HTML(f"""
            <div style='padding: 20px; background: #f0f8ff; border-radius: 10px; border-left: 5px solid #1F4788;'>
                <h3>🤖 Multi-Agent System Processing...</h3>
                <p style='color: #666;'>Question: <strong>{query}</strong></p>
            </div>
            """))
            
            try:
                # Execute query
                result = executor.execute(query, show_viz=True)
                
                if result['success']:
                    # Show query classification
                    display(HTML(f"""
                    <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; margin: 10px 0;'>
                        <strong>📋 Query Classification:</strong> {result['query_type']}
                    </div>
                    """))
                    
                    # Show SQL
                    display(Markdown("### 🔍 Generated SQL"))
                    display(HTML(f"""
                    <div style='background: #f5f5f5; padding: 15px; border-radius: 8px; overflow-x: auto;'>
                        <pre style='margin: 0;'>{result['sql']}</pre>
                    </div>
                    """))
                    
                    # Show insights
                    display(HTML(f"""
                    <div style='padding: 15px; background: #fff3e0; border-radius: 8px; margin: 10px 0;'>
                        <strong>💡 Insights:</strong> {result['insights']}
                    </div>
                    """))
                    
                    # Show data table
                    display(Markdown("### 📊 Results"))
                    display(HTML(f"<p><strong>Retrieved {result['row_count']} rows</strong></p>"))
                    
                    if result['row_count'] > 0:
                        # Show top 10 rows
                        display_df = result['dataframe'].head(10)
                        display(display_df.style.set_properties(**{
                            'background-color': '#f9f9f9',
                            'border-color': '#ddd',
                            'text-align': 'left'
                        }).set_table_styles([{
                            'selector': 'th',
                            'props': [('background-color', '#1F4788'), ('color', 'white'), 
                                     ('font-weight', 'bold'), ('text-align', 'left')]
                        }]))
                        
                        if result['row_count'] > 10:
                            display(HTML(f"<p><em>Showing first 10 of {result['row_count']} rows</em></p>"))
                    
                    # Show visualization
                    if result['visualization']:
                        display(Markdown("### 📈 Visualization"))
                        result['visualization'].show()
                    
                    # Success message
                    display(HTML("""
                    <div style='padding: 15px; background: #e8f5e9; border-radius: 8px; margin: 20px 0;'>
                        <h3 style='color: #2e7d32; margin: 0;'>✅ Query Completed Successfully!</h3>
                    </div>
                    """))
                    
                else:
                    # Show error
                    display(HTML(f"""
                    <div style='padding: 20px; background: #ffebee; border-radius: 10px; border-left: 5px solid #c62828;'>
                        <h3 style='color: #c62828;'>❌ Error</h3>
                        <p>{result['error']}</p>
                    </div>
                    """))
            
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 20px; background: #ffebee; border-radius: 10px; border-left: 5px solid #c62828;'>
                    <h3 style='color: #c62828;'>❌ Unexpected Error</h3>
                    <p>{str(e)}</p>
                </div>
                """))
    
    button.on_click(on_click)
    return button

test_buttons = [create_test_button(q) for q in test_queries]
display(widgets.VBox(test_buttons))

---

## 🔧 Advanced: Direct Executor Access

For programmatic access or batch processing:

In [28]:
# Example: Run a query programmatically
question = "Show me top 5 markets by sends"

print(f"💬 Question: {question}\n")

result = executor.execute(question)

if result['success']:
    print(f"✅ Query Type: {result['query_type']}")
    print(f"📊 Rows: {result['row_count']}")
    print(f"\n💡 Insights: {result['insights']}")
    print(f"\n📋 Data:")
    print(result['dataframe'].to_string(index=False))
    
    if result['visualization']:
        result['visualization'].show()
else:
    print(f"❌ Error: {result['error']}")

💬 Question: Show me top 5 markets by sends

   [DEBUG] Raw LLM output:  SELECT BUSINESSUNIT, SUM(SENDS) as total_sends
    FROM V_FACT_SFMC_SEND_PERFORMANCE_TRACKING
    GROUP BY BUSINESSUNIT
    ORDER BY total_sends DESC
    LIMIT 5;...
   [DEBUG] Clean SQL: SELECT BUSINESSUNIT, SUM(SENDS) as total_sends
    FROM V_FACT_SFMC_SEND_PERFORMANCE_TRACKING
    GROUP BY BUSINESSUNIT
    ORDER BY total_sends DESC
    LIMIT 5...
✅ Query Type: RANKING
📊 Rows: 5

💡 Insights: Top performer: VCDK | Total: 340,402,955 | Average: 68,080,591.00

📋 Data:
BUSINESSUNIT  TOTAL_SENDS
        VCDK     74148406
        VCJP     68384748
        VCCH     67138253
        VCSE     66354033
        VCGR     64377515


---

## 📊 System Architecture

```
User Question (Natural Language)
        ↓
┌─────────────────────────┐
│ MultiAgentExecutor      │
│  ├─ Query Classifier    │ → Determines query type
│  ├─ CortexAnalyst       │ → Generates SQL via LLM
│  ├─ SQL Execution       │ → Runs query on Snowflake
│  ├─ VisualizationAgent  │ → Creates appropriate chart
│  └─ Insight Generator   │ → Extracts key findings
└─────────────────────────┘
        ↓
Results + Visualization
```

### Key Features:
- 🤖 **Natural Language Processing**: Ask questions like you would to a colleague
- 🎯 **Intelligent Routing**: Automatic classification of query type
- 📊 **Auto-Visualization**: Charts generated based on data and query type
- 💡 **Insight Generation**: Automatic summary of key findings
- 🔒 **Governed**: All within Snowflake security boundaries